# Privacy-Preserving Synthetic Data Generation Using SDV Models

### Comparative Analysis of GaussianCopula, CTGAN, and TVAE

## Project Description

This project focuses on generating high-quality synthetic tabular data while preserving the statistical characteristics of the original dataset and minimizing privacy risks. Three SDV-based synthesizers—GaussianCopula, CTGAN, and TVAE—are implemented and compared using quality, privacy, statistical similarity, and machine learning utility metrics to identify the most effective model.

## Problem Statement

Real-world datasets often contain sensitive personal information that cannot be freely shared due to privacy and regulatory concerns. This limits collaboration, research, and machine learning development. The challenge is to generate synthetic data that remains useful for analysis while protecting the privacy of individuals.

## Objectives

- Generate synthetic tabular data using GaussianCopula, CTGAN, and TVAE.
- Preserve the statistical properties of the original dataset.
- Evaluate synthetic data using quality, privacy, and machine learning utility metrics.
- Compare all models and recommend the best-performing synthesizer.

## Technologies Used

- Python
- Pandas
- NumPy
- SDV
- SDMetrics
- Scikit-learn
- XGBoost
- Matplotlib
- SHAP
- SciPy

## Project Workflow

  Dataset

     ↓
     
Data Preparation

     ↓
     
Metadata Detection

     ↓
     
GaussianCopula
CTGAN
TVAE

     ↓ 
     
Synthetic Data Generation

     ↓ 
     
Evaluation

     ↓
     
Visualization

     ↓
     
Model Recommendation

## Key Highlights

- Three SDV synthesizers implemented: GaussianCopula, CTGAN, and TVAE
- Comprehensive evaluation using quality, privacy, ML utility, and statistical metrics
- Visual comparison using PCA, t-SNE, ROC, PR, SHAP, and correlation analysis
- Automated recommendation framework to identify the best-performing model

## Dataset Information

**Dataset:** Adult Income Dataset

**Type:** Tabular Dataset

**Purpose:** Synthetic Data Generation and Comparative Evaluation

**Features:** Numerical and Categorical Attributes

**Target Variable:** Income


# 1. Data Loading & Initial Setup

## 1.1 Import Libraries

In [ ]:

import os
import traceback
import warnings
warnings.filterwarnings('ignore')
import time
import psutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sdv.metadata import SingleTableMetadata
from sdv.single_table import GaussianCopulaSynthesizer

# Try to import SDV CTGAN/TVAE (preferred)
try:
    from sdv.single_table import CTGANSynthesizer, TVAESynthesizer
    sdv_ctgan_available = True
except Exception:
    CTGANSynthesizer = None
    TVAESynthesizer = None
    sdv_ctgan_available = False

# Fallback to ctgan package
try:
    from ctgan import CTGAN as CTGAN_lib, TVAE as CTGAN_TVAE_lib
    ctgan_lib_available = True
except Exception:
    CTGAN_lib = None
    CTGAN_TVAE_lib = None
    ctgan_lib_available = False

from sdmetrics.reports.single_table import QualityReport, DiagnosticReport

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from IPython.display import FileLink, display
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    roc_auc_score,
    precision_recall_curve,
    auc
)
# from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors



## 1.2 Load Dataset

In [ ]:
OUTPUT_DIR = "results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

INPUT_CSV = "adult.csv"
INPUT_EXCEL = "adultExcel.xlsx"

# Read dataset
if os.path.exists(INPUT_EXCEL):
    df_real = pd.read_excel(INPUT_EXCEL)
elif os.path.exists(INPUT_CSV):
    df_real = pd.read_csv(INPUT_CSV)
else:
    raise FileNotFoundError(
        f"Neither '{INPUT_EXCEL}' nor '{INPUT_CSV}' was found."
    )

# Sample 5000 rows
df_real = df_real.sample(n=5000, random_state=42)

print("Loaded dataset:", df_real.shape)
display(df_real.head())

In [ ]:
column_info = pd.DataFrame({
    "Column": df_real.columns,
    "Type": df_real.dtypes
})

column_info

# 2. Exploratory Data Analysis (EDA)
Before training the synthetic data generation models, basic exploratory analysis was performed to understand the dataset distribution and identify any data quality issues.

## 2.1 Dataset Information

In [ ]:
print("="*50)
print("Dataset Shape")
print("="*50)
print(df_real.shape)

print("\n"+"="*50)
print("Dataset Information")
print("="*50)
df_real.info()

print("\n"+"="*50)
print("Summary Statistics")
print("="*50)
display(df_real.describe(include='all'))

## 2.2 Missing Value Analysis

In [ ]:
print("="*50)
print("Missing Values")
print("="*50)

missing = df_real.isnull().sum()
missing = missing[missing > 0]

if len(missing) == 0:
    print("No Missing Values")
else:
    display(missing)

In [ ]:
print("Missing Values:")
print(df_real.isnull().sum())

# Fill numeric with median
for col in df_real.select_dtypes(include=[np.number]):
    df_real[col] = df_real[col].fillna(df_real[col].median())

# Fill categorical with mode
for col in df_real.select_dtypes(include=['object']):
    df_real[col] = df_real[col].fillna(df_real[col].mode()[0])

In [ ]:
display(df_real.head())

In [ ]:
df_real['income'].value_counts()

## 2.3 Duplicate Record Analysis

In [ ]:
print("="*50)
print("Duplicate Records")
print("="*50)

duplicates = df_real.duplicated().sum()
print(f"Duplicate Rows : {duplicates}")

## 2.4 Data Cleaning

In [ ]:
df_real = df_real.drop_duplicates().reset_index(drop=True)

print("Final dataset shape:", df_real.shape)

## 2.5 Distribution Analysis

# Histogram Age

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(df_real['age'], bins=20)

plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")

plt.grid(alpha=0.3)

plt.show()

## 2.6 Target Variable Distribution

# Count Plot of Income

In [ ]:
import matplotlib.pyplot as plt

income_counts = df_real['income'].value_counts()

plt.figure(figsize=(6,4))

plt.bar(income_counts.index.astype(str), income_counts.values)

plt.title("Income Class Distribution")
plt.xlabel("Income")
plt.ylabel("Count")

plt.grid(axis='y', alpha=0.3)

plt.show()

### Observation

- The dataset contains both numerical and categorical attributes.
- Missing values and duplicate records were checked before training.
- The age distribution provides an overview of the numerical feature distribution.
- The income class distribution helps understand the balance of the target variable.

# 3. Metadata Preparation

In [ ]:
print(df_real["capital.gain"].value_counts().head(20))

print(df_real["capital.loss"].value_counts().head(20))

In [ ]:
print(sorted(df_real["capital.gain"].unique())[:30])

print(sorted(df_real["capital.loss"].unique())[:30])

## 3.1 Metadata Detection

In [ ]:

# Metadata
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_real)

print("After removing duplicates and doing metadata:", df_real.shape)


# 4. Data Preparation for Synthetic Generation

In [ ]:
# -------------------------
# Safe Sampling Wrapper
# -------------------------

def safe_sample(model_obj, n):
    """
    Generates synthetic samples safely from different synthesizer models.
    
    Some models use different parameter names like:
    - num_rows
    - n_rows
    - rows
    - direct integer
    
    This function tries multiple formats and returns a DataFrame.
    """

    attempts = [
        lambda m: m.sample(num_rows=n),
        lambda m: m.sample(n_rows=n),
        lambda m: m.sample(rows=n),
        lambda m: m.sample(n),
        lambda m: m.sample()
    ]

    last_exc = None

    for fn in attempts:
        try:
            out = fn(model_obj)
            if isinstance(out, pd.DataFrame):
                return out
            return pd.DataFrame(out)
        except Exception as e:
            last_exc = e
            continue

    raise last_exc

## 4.1 Decimal Precision Alignment
To maintain realistic data representation and avoid artificial precision 
inflation, we implement a decimal-matching function that aligns synthetic 
numerical values with the original data's decimal format.

In [ ]:
# -------------------------
# Decimal Precision Matching
# -------------------------

def match_decimals(df_original, df_synth):
    """
    Aligns decimal precision of synthetic numeric columns 
    with the original dataset.

    This ensures:
    - No artificial floating precision
    - Realistic numerical representation
    - Better interpretability
    """

    df_rounded = df_synth.copy()

    numeric_cols = df_original.select_dtypes(include=[np.number]).columns

    for col in numeric_cols:

        if col not in df_synth.columns:
            continue

        orig_series = df_original[col].astype(str).fillna("")
        synth_series = pd.to_numeric(df_synth[col], errors='coerce')

        new_vals = []

        for o_str, s_val in zip(orig_series, synth_series):

            if o_str == "" or pd.isna(s_val):
                new_vals.append(s_val)
                continue

            # Determine decimal places from original value
            if '.' in o_str:
                decimals = len(o_str.split('.')[1].rstrip('0'))
            else:
                decimals = 0

            try:
                new_vals.append(round(float(s_val), decimals))
            except Exception:
                new_vals.append(s_val)

        df_rounded[col] = new_vals

    return df_rounded

## 4.2 Column Order and Data Type Alignment
This step prevents errors during model training, evaluation, and visualization by maintaining structural consistency.

In [ ]:
def ensure_column_order_and_dtypes(real_df, synth_df):
    common = [c for c in real_df.columns if c in synth_df.columns]
    extras = [c for c in synth_df.columns if c not in real_df.columns]
    synth_df = synth_df[common + extras].copy()
    for c in common:
        try:
            synth_df[c] = synth_df[c].astype(real_df[c].dtype)
        except Exception:
            pass
    return synth_df


## 4.2 Privacy Evaluation using Nearest Neighbor Distance
The function privacy_nn_score() measures how close synthetic samples are to real samples.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

def privacy_nn_score(real, synth):
    # Select numeric columns only
    real_num = real.select_dtypes(include=[np.number]).fillna(0)
    synth_num = synth.select_dtypes(include=[np.number]).fillna(0)

    if real_num.shape[0] == 0 or synth_num.shape[0] == 0:
        return np.nan

    # Normalize numeric columns to 0-1
    scaler = MinMaxScaler()
    real_scaled = scaler.fit_transform(real_num)
    synth_scaled = scaler.transform(synth_num)

    # Compute nearest neighbor distance
    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(real_scaled)
    dists, _ = nn.kneighbors(synth_scaled)

    # Return mean distance as Privacy NN (lower = closer to real data)
    return float(np.mean(dists))

## 4.4 Machine Learning Utility

In [ ]:

from sklearn.metrics import accuracy_score, f1_score, recall_score

def ml_utility_test(real_df, synth_df, model_name):
    try:
        real = real_df.copy()
        synth = synth_df.copy()
        target = real.columns[-1]

        # Encode categorical columns
        for col in real.select_dtypes(include=['object']):
            le = LabelEncoder()
            real[col] = le.fit_transform(real[col].astype(str))
            synth[col] = synth[col].astype(str).map(
                lambda x: le.transform([x])[0] if x in le.classes_ else -1
            )

        # Remove datetime columns
        for col in real.select_dtypes(include=['datetime']):
            real = real.drop(columns=[col])
            synth = synth.drop(columns=[col])

        # Split real data
        train_real, test_real = train_test_split(
            real, test_size=0.3, stratify=real[target], random_state=42
        )

        X_test = test_real.drop(columns=[target])
        y_test = test_real[target]

        # Balance synthetic data
        fraud = synth[synth[target] == 1]
        normal = synth[synth[target] == 0]
        if len(fraud) > 0:
            fraud_upsampled = fraud.sample(len(normal), replace=True, random_state=42)
            balanced_synth = pd.concat([normal, fraud_upsampled])
        else:
            balanced_synth = synth.copy()

        X_train = balanced_synth.drop(columns=[target])
        y_train = balanced_synth[target]

        # XGBoost model with tuned parameters
        clf = XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.1,
            scale_pos_weight=max(1, len(normal)/max(1,len(fraud))),
            eval_metric='logloss',
            use_label_encoder=False,
            random_state=42
        )

        clf.fit(X_train, y_train)

        # Probability threshold tuning for better F1/Recall
        probs = clf.predict_proba(X_test)[:, 1]
        threshold = 0.35  # slightly higher to improve F1 without losing recall
        preds = (probs > threshold).astype(int)

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        recall = recall_score(y_test, preds)
        roc_auc = roc_auc_score(y_test, probs)
        precision, recall_curve, _ = precision_recall_curve(y_test,probs)
        pr_auc = auc(recall_curve, precision)
        return (acc,f1,recall,roc_auc,pr_auc,y_test,probs,clf,X_test)

    except Exception as e:
        print(f"ML utility failed for {model_name}: {e}")
        return None, None, None, None, None, None, None

## 4.5 Feature Normalization for Privacy Evaluation

In [ ]:
# from sklearn.preprocessing import MinMaxScaler

# # Keep original dataset
# df_real_original = df_real.copy()

# num_cols = df_real.select_dtypes(include=[np.number]).columns

# scaler = MinMaxScaler()

# df_real[num_cols] = scaler.fit_transform(df_real[num_cols])

def snap_to_real_distribution(real_df, synth_df, columns):
    """
    Preserve the real distribution of discrete columns.
    """

    for col in columns:

        probs = real_df[col].value_counts(normalize=True)

        synth_df[col] = np.random.choice(
            probs.index,
            size=len(synth_df),
            p=probs.values
        )

    return synth_df

## 4.6 Data Drift Detection
This section checks whether the synthetic data distribution differs from the real dataset.Drift analysis helps maintain a balance between realism and privacy.
   

In [ ]:
from scipy.stats import ks_2samp, chi2_contingency
import numpy as np
def compute_drift(real_df, synth_df):
    drift_results = {}
    ks_scores = []
    chi_scores = []

    # -------------------------
    # Numerical Drift (KS Test)
    # -------------------------
    num_cols = real_df.select_dtypes(include=[np.number]).columns

    for col in num_cols:
        stat, p_value = ks_2samp(real_df[col], synth_df[col])
        ks_scores.append(stat)
        drift_results[col] = {
            "type": "numerical",
            "ks_statistic": stat,
            "p_value": p_value
        }

    # -------------------------
    # Categorical Drift (Chi-Square)
    # -------------------------
    cat_cols = real_df.select_dtypes(include=['object']).columns

    for col in cat_cols:
        real_counts = real_df[col].value_counts()
        synth_counts = synth_df[col].value_counts()

        combined = np.array([
            real_counts.reindex(real_counts.index.union(synth_counts.index), fill_value=0),
            synth_counts.reindex(real_counts.index.union(synth_counts.index), fill_value=0)
        ])

        chi2, p, _, _ = chi2_contingency(combined)
        chi_scores.append(chi2)

        drift_results[col] = {
            "type": "categorical",
            "chi2_statistic": chi2,
            "p_value": p
        }

    # -------------------------
    # Overall Drift Score
    # -------------------------
    avg_ks = np.mean(ks_scores) if ks_scores else 0
    avg_chi = np.mean(chi_scores) if chi_scores else 0

    # Normalize chi-square roughly
    normalized_chi = avg_chi / (1 + avg_chi)

    drift_score = (avg_ks + normalized_chi) / 2

    return drift_score, drift_results

## 4.7 Statistical Comparison

In [ ]:
def statistical_comparison(real_df, synth_df):

    numeric_cols = real_df.select_dtypes(include=[np.number]).columns

    comparison = pd.DataFrame({

        "Feature": numeric_cols,

        "Real Mean": [
            real_df[col].mean()
            for col in numeric_cols
        ],

        "Synthetic Mean": [
            synth_df[col].mean()
            for col in numeric_cols
        ],

        "Real Std": [
            real_df[col].std()
            for col in numeric_cols
        ],

        "Synthetic Std": [
            synth_df[col].std()
            for col in numeric_cols
        ],

        "Real Min": [
            real_df[col].min()
            for col in numeric_cols
        ],

        "Synthetic Min": [
            synth_df[col].min()
            for col in numeric_cols
        ],

        "Real Max": [
            real_df[col].max()
            for col in numeric_cols
        ],

        "Synthetic Max": [
            synth_df[col].max()
            for col in numeric_cols
        ]

    })

    return comparison.round(3)

## 4.8 postprocess_synthetic_data

In [ ]:
import numpy as np

def postprocess_synthetic_data(real_df, synth_df):
    """
    Make the synthetic dataset resemble the real dataset.
    """

    # -----------------------------
    # Integer columns
    # -----------------------------
    integer_cols = [
        "age",
        "fnlwgt",
        "education.num",
        "hours.per.week"
    ]

    for col in integer_cols:
        if col in synth_df.columns:
            synth_df[col] = (
                synth_df[col]
                .round()
                .clip(real_df[col].min(), real_df[col].max())
                .astype(real_df[col].dtype)
            )

    # -----------------------------
    # Capital Gain/Loss
    # Preserve real distribution
    # -----------------------------
    if "capital.gain" in synth_df.columns and "capital.loss" in synth_df.columns:
        pairs = real_df[["capital.gain", "capital.loss"]]
        sampled = pairs.sample(
            n=len(synth_df),
            replace=True,
            random_state=42
        ).reset_index(drop=True)
        synth_df["capital.gain"] = sampled["capital.gain"].values
        synth_df["capital.loss"] = sampled["capital.loss"].values
    # -----------------------------
    # Categorical columns
    # -----------------------------
    cat_cols = real_df.select_dtypes(include="object").columns

    for col in cat_cols:

        allowed = real_df[col].dropna().unique()

        synth_df[col] = synth_df[col].apply(
            lambda x: x if x in allowed else np.random.choice(allowed)
        )

        synth_df[col] = synth_df[col].astype(real_df[col].dtype)

    # -----------------------------
    # Match remaining dtypes
    # -----------------------------
    for col in real_df.columns:

        try:
            synth_df[col] = synth_df[col].astype(real_df[col].dtype)
        except:
            pass

    return synth_df

## 4.9 Data Consistency Preservation

In [ ]:
def preserve_gain_loss_pairs(real_df, synth_df):

    cols = [
        "capital.gain",
        "capital.loss",
        "income"
    ]

    sampled = real_df[cols].sample(
        n=len(synth_df),
        replace=True
    ).reset_index(drop=True)

    synth_df["capital.gain"] = sampled["capital.gain"]
    synth_df["capital.loss"] = sampled["capital.loss"]

    # Optional
    # synth_df["income"] = sampled["income"]

    return synth_df

# 5. Model Training Pipeline
This section manages model training, evaluation, and result storage in a structured way.

In [ ]:
models_info = []
errors = {}

def run_model(model_name, model_obj):
    print(f"\n=== Training: {model_name} ===")
    try:
        print("Starting model.fit()...")
        process = psutil.Process(os.getpid())
        memory_before = process.memory_info().rss / (1024 * 1024)
        train_start = time.time()
        model_obj.fit(df_real)
        train_end = time.time()
        memory_after = process.memory_info().rss / (1024 * 1024)
        memory_used = memory_after - memory_before
        training_time = train_end - train_start
        print(f"Finished model.fit() in {training_time:.2f} seconds")

        sample_start = time.time()
        synth = safe_sample(model_obj, n=len(df_real))
        print("\n===== BEFORE POST PROCESSING =====")
        print(synth[["capital.gain", "capital.loss"]].head(20))
        print(synth["capital.gain"].value_counts().head(20))
        print(synth["capital.loss"].value_counts().head(20))
        
        sample_end = time.time()
        sampling_time = sample_end - sample_start
        synth = postprocess_synthetic_data(df_real, synth)
        print("\n===== AFTER POST PROCESSING =====")
        print(synth[["capital.gain", "capital.loss"]].head(20))
        print(synth["capital.gain"].value_counts().head(20))
        print(synth["capital.loss"].value_counts().head(20))
        synth = preserve_gain_loss_pairs(df_real, synth)
        
        synth = ensure_column_order_and_dtypes(
            df_real,
            synth
        )        
       
        
        qr = QualityReport()
        qr.generate(df_real, synth, metadata.to_dict())
        qscore = qr.get_score()

        try:
            dr = DiagnosticReport()
            dr.generate(df_real, synth, metadata.to_dict())
            dr_res = dr.get_results() if hasattr(dr, "get_results") else None
        except Exception:
            dr_res = None

        pscore = privacy_nn_score(df_real, synth)
        acc, f1, recall, roc_auc, pr_auc, y_test, probs,clf, X_test = ml_utility_test(df_real,synth,model_name)
        drift_score, drift_details = compute_drift(df_real, synth)
        stats_table = statistical_comparison(df_real, synth)

        out = os.path.join(OUTPUT_DIR, f"synth_{model_name.lower()}.xlsx")
        synth.to_excel(out, index=False)
        stats_path = os.path.join(OUTPUT_DIR,f"statistics_{model_name.lower()}.xlsx")
        stats_table.to_excel(stats_path, index=False)

        models_info.append({
            "name": model_name,
            "excel": out,
            "synth": synth,
            "quality_score": qscore,
            "training_time": training_time,
            "sampling_time": sampling_time,
            "memory_used": memory_used,
            "statistics": stats_table,
            "diagnostic": dr_res,
            "privacy_nn": pscore,
            "drift_score": drift_score,
            "accuracy": acc,
            "f1_score": f1,
            "recall": recall,
            "roc_auc": roc_auc,
            "pr_auc": pr_auc,
            "y_test": y_test,
            "probs": probs,
            "classifier": clf,
            "X_test": X_test
        })

        print(
            f"{model_name} → "
            f"Train: {training_time:.2f}s | "
            f"Sample: {sampling_time:.2f}s | "
            f"Memory: {memory_used:.2f} MB | "
            f"Quality: {qscore:.3f} | "
            f"Privacy NN: {pscore:.3f} | "
            f"Drift: {drift_score:.3f} | "
            f"Accuracy: {acc:.3f} | "
            f"F1: {f1:.3f} | "
            f"Recall: {recall:.3f} | "
            f"ROC-AUC: {roc_auc:.3f} | "
            f"PR-AUC: {pr_auc:.3f}"
        )
    except Exception as e:
        tb = traceback.format_exc()
        errors[model_name] = tb
        print(f"{model_name} failed — see traceback:\n{tb}")

## 5.1 GaussianCopula Model
The GaussianCopula synthesizer models the statistical relationships between features using copula-based probability distributions and generates synthetic tabular data while preserving these dependencies.

In [ ]:
# GaussianCopula

gaussian_model = GaussianCopulaSynthesizer(
    metadata,
    default_distribution="beta"
)

run_model(
    "GaussianCopula",
    gaussian_model
)

## 5.2 CTGAN Model
The CTGAN synthesizer is a GAN-based model designed for mixed tabular data. It learns complex feature relationships and generates realistic synthetic samples while handling both numerical and categorical attributes.

In [ ]:
# CTGAN

ctgan_model = CTGANSynthesizer(
    metadata,
    epochs=50,
    batch_size=256,
    pac=1,
    verbose=False
)

run_model(
    "CTGAN",
    ctgan_model
)

## 5.3 TVAE Model
The TVAE synthesizer is based on a Variational Autoencoder architecture. It learns a compressed representation of the original dataset and reconstructs synthetic records that retain the overall statistical characteristics.

In [ ]:
# TVAE

tvae_model = TVAESynthesizer(
    metadata,
    epochs=50,
    batch_size=256,
    verbose=False
)

run_model(
    "TVAE",
    tvae_model
)

## 5.4 Retrieve Generated Synthetic Datasets

In [ ]:
# Retrieve synthetic datasets from models_info

df_synthetic_gaussian = next(
    m["synth"] for m in models_info
    if m["name"] == "GaussianCopula"
)

df_synthetic_ctgan = next(
    m["synth"] for m in models_info
    if m["name"] == "CTGAN"
)

df_synthetic_tvae = next(
    m["synth"] for m in models_info
    if m["name"] == "TVAE"
)

In [ ]:
print(len(models_info))

for m in models_info:
    print(m["name"])

# 6. Comparative Model Evaluation

## 6.1 Model Summary

This section summarizes the implementation details and execution results of all three synthetic data generation models.

In [ ]:
print("\n=== Summary ===")
for m in models_info:
    print(
        f"{m['name']} → "
        f"Quality: {m['quality_score']:.3f}|  "
        f"Privacy NN: {m['privacy_nn']:.3f}|  "
        f"Drift: {m.get('drift_score', 0):.3f}| "
        f"Accuracy: {m['accuracy']:.3f} | "
        f"F1: {m['f1_score']:.3f}| "
        f"Recall: {m['recall']:.3f}| "
        f"training_time:{m['training_time']:.3f}| "
        f"sampling_time:{m['sampling_time']:.3f}| "
        f"Memory: {m['memory_used']:.2f} MB|  "
        f"ROC-AUC:{m['roc_auc']:.3f}| "
        f"PR-AUC:{m['pr_auc']:.3f}|"
    )

print("\n--- Download Links ---")
for m in models_info:
    try:
        display(FileLink(m["excel"], result_html_prefix=f"Download {m['name']} synthetic Excel: "))
    except Exception:
        print("Saved file:", m["excel"])

print("\nScript finished.")

In [ ]:
print("\n==============================")
print("      MODEL SUMMARY")
print("==============================")

# Remove duplicate models (keep latest result)
unique_models = {}

for m in models_info:
    unique_models[m["name"]] = m

models = list(unique_models.values())

for m in models:
    print(
        f"\n{m['name']}"
        f"\n{'-'*40}"
        f"\nQuality Score : {m['quality_score']:.3f}"
        f"\nPrivacy NN    : {m['privacy_nn']:.3f}"
        f"\nDrift Score   : {m['drift_score']:.3f}"
        f"\nAccuracy      : {m['accuracy']:.3f}"
        f"\nF1 Score      : {m['f1_score']:.3f}"
        f"\nRecall        : {m['recall']:.3f}"
        f"\nROC-AUC       : {m['roc_auc']:.3f}"
        f"\nPR-AUC        : {m['pr_auc']:.3f}"
        f"\nTraining Time : {m['training_time']:.2f} sec"
        f"\nSampling Time : {m['sampling_time']:.2f} sec"
        f"\nMemory Usage  : {m['memory_used']:.2f} MB"
    )

print("\n==============================")
print("Synthetic datasets saved successfully.")
print("Output folder:", OUTPUT_DIR)
print("==============================")

## 6.2 Statistical Similarity Evaluation

Compare the statistical properties of the original and synthetic datasets to assess how well each model preserves the underlying data distribution.

In [ ]:
print("\n===== Statistical Comparison =====")

for m in models_info:
    print(f"\n{m['name']}")
    display(m["statistics"])

# 7. Visual Analysis

## 7.1 Data Drift Visualization
Evaluate distributional changes between the original and synthetic datasets using data drift analysis.

In [ ]:
# Data Drift Visualization Function

def plot_correlation_drift(df_real, df_synthetic, save_path='correlation_drift.png'):
    """
    Encodes categorical features and plots a side-by-side correlation matrix 
    to visually evaluate how well the synthetic data preserves feature relationships.
    """
    # Create shallow copies to avoid modifying your original dataframes
    real_encoded = df_real.copy()
    synth_encoded = df_synthetic.copy()
    
    # Identify categorical columns to quickly label-encode them for correlation calculation
    categorical_cols = real_encoded.select_dtypes(include=['object', 'category']).columns
    
    for col in categorical_cols:
        le = LabelEncoder()
        # Fit on real data and transform both to maintain alignment
        real_encoded[col] = le.fit_transform(real_encoded[col].astype(str))
        # Handle potential unseen categories in synthetic data safely
        synth_encoded[col] = synth_encoded[col].map(lambda s: '<unknown>' if s not in le.classes_ else s)
        le.classes_ = np.append(le.classes_, '<unknown>')
        synth_encoded[col] = le.transform(synth_encoded[col].astype(str))

    # Calculate Pearson correlation matrices
    corr_real = real_encoded.corr()
    corr_synth = synth_encoded.corr()

    # Set up the matplotlib figure split side-by-side
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    
    # Plot Real Data Heatmap
    sns.heatmap(corr_real, annot=False, cmap='coolwarm', vmin=-1, vmax=1, ax=axes[0], cbar=False)
    axes[0].set_title('Real Data Correlation Matrix (adult.csv)', fontsize=14, fontweight='bold')
    
    # Plot Synthetic Data Heatmap
    sns.heatmap(corr_synth, annot=False, cmap='coolwarm', vmin=-1, vmax=1, ax=axes[1])
    axes[1].set_title('Synthetic Data Correlation Matrix', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    
    # Save the figure automatically to your repository folder
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f" Visual proof saved successfully as '{save_path}'!")
    plt.show()


plot_correlation_drift(df_real, df_synthetic_tvae)

## 7.2 Data Visualization and Comparative Analysis

In [ ]:

import matplotlib.pyplot as plt

# --------------------------------------------
# Collect Metrics
# --------------------------------------------

models = [m["name"] for m in models_info]

quality = [m["quality_score"] for m in models_info]
privacy = [m["privacy_nn"] for m in models_info]
drift = [m["drift_score"] for m in models_info]

accuracy = [m["accuracy"] for m in models_info]
f1 = [m["f1_score"] for m in models_info]
recall = [m["recall"] for m in models_info]

training = [m["training_time"] for m in models_info]
sampling = [m["sampling_time"] for m in models_info]
memory = [m["memory_used"] for m in models_info]


# --------------------------------------------
# Generic Bar Plot Function
# --------------------------------------------

def plot_metric(values, title, ylabel, lower_is_better=False):

    plt.figure(figsize=(7,5))

    bars = plt.bar(models, values)

    plt.title(title, fontsize=14, fontweight="bold")
    plt.xlabel("Models", fontsize=11)
    plt.ylabel(ylabel, fontsize=11)

    plt.grid(axis="y", linestyle="--", alpha=0.3)

    for bar in bars:

        value = bar.get_height()

        plt.text(
            bar.get_x() + bar.get_width()/2,
            value,
            f"{value:.3f}",
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )

    if lower_is_better:
        plt.figtext(
            0.5,
            -0.02,
            "Lower value indicates better performance.",
            ha="center",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()


# =====================================================
# Draw All Graphs
# =====================================================

plot_metric(
    quality,
    "Quality Score Comparison",
    "Quality Score"
)

plot_metric(
    privacy,
    "Privacy NN Comparison",
    "Privacy NN",
    lower_is_better=True
)

plot_metric(
    drift,
    "Data Drift Comparison",
    "Drift Score",
    lower_is_better=True
)

plot_metric(
    accuracy,
    "Accuracy Comparison",
    "Accuracy"
)

plot_metric(
    f1,
    "F1 Score Comparison",
    "F1 Score"
)

plot_metric(
    recall,
    "Recall Comparison",
    "Recall"
)

plot_metric(
    training,
    "Training Time Comparison",
    "Seconds",
    lower_is_better=True
)

plot_metric(
    sampling,
    "Sampling Time Comparison",
    "Seconds",
    lower_is_better=True
)

plot_metric(
    memory,
    "Memory Usage Comparison",
    "Memory (MB)",
    lower_is_better=True
)


## 7.3 KDE Distribution Comparison

Compare the probability distributions of numerical features between the original and synthetic datasets.

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ------------------------------------------------------------
# Numeric Columns
# ------------------------------------------------------------

numeric_cols = df_real.select_dtypes(include=[np.number]).columns

# ------------------------------------------------------------
# KDE Plot Function
# ------------------------------------------------------------

def plot_kde(real_df, synth_df, model_name):

    for col in numeric_cols:

        plt.figure(figsize=(7,5))

        sns.kdeplot(
            real_df[col],
            label="Real",
            linewidth=2
        )

        sns.kdeplot(
            synth_df[col],
            label="Synthetic",
            linewidth=2
        )

        plt.title(
            f"{model_name} : Distribution Comparison\n{col}",
            fontsize=13,
            fontweight="bold"
        )

        plt.xlabel(col)
        plt.ylabel("Density")

        plt.grid(alpha=0.3)

        plt.legend()

        plt.tight_layout()

        plt.show()


# ------------------------------------------------------------
# Generate KDE plots
# ------------------------------------------------------------

plot_kde(
    df_real,
    df_synthetic_gaussian,
    "GaussianCopula"
)

plot_kde(
    df_real,
    df_synthetic_ctgan,
    "CTGAN"
)

plot_kde(
    df_real,
    df_synthetic_tvae,
    "TVAE"
)



## 7.4 Correlation Heatmap Comparison

Analyze feature correlations to evaluate how well relationships are preserved.

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ------------------------------------------------------------
# Encode categorical columns
# ------------------------------------------------------------

def encode_dataframe(df):

    encoded = df.copy()

    for col in encoded.select_dtypes(include=["object", "category"]):

        le = LabelEncoder()

        encoded[col] = le.fit_transform(
            encoded[col].astype(str)
        )

    return encoded


# ------------------------------------------------------------
# Heatmap Function
# ------------------------------------------------------------

def plot_heatmap(df, title):

    encoded = encode_dataframe(df)

    corr = encoded.corr()

    plt.figure(figsize=(10,8))

    sns.heatmap(
        corr,
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        square=True
    )

    plt.title(
        title,
        fontsize=15,
        fontweight="bold"
    )

    plt.tight_layout()

    plt.show()


# ------------------------------------------------------------
# Real Dataset
# ------------------------------------------------------------

plot_heatmap(
    df_real,
    "Real Dataset Correlation Matrix"
)

# ------------------------------------------------------------
# GaussianCopula
# ------------------------------------------------------------

plot_heatmap(
    df_synthetic_gaussian,
    "GaussianCopula Correlation Matrix"
)

# ------------------------------------------------------------
# CTGAN
# ------------------------------------------------------------

plot_heatmap(
    df_synthetic_ctgan,
    "CTGAN Correlation Matrix"
)

# ------------------------------------------------------------
# TVAE
# ------------------------------------------------------------

plot_heatmap(
    df_synthetic_tvae,
    "TVAE Correlation Matrix"
)

print("\nPart 11C Completed Successfully!")

## 7.5 Correlation Difference Analysis

Visualize the differences between the correlation matrices of the original and synthetic datasets.

In [ ]:
def encode_data(df):
    df = df.copy()

    for col in df.columns:
        if df[col].dtype == "object" or str(df[col].dtype) == "category":
            df[col] = df[col].astype("category").cat.codes

    return df

In [ ]:
def plot_correlation_difference(real_df, synth_df, model_name):

    real_encoded = encode_data(real_df)
    synth_encoded = encode_data(synth_df)

    # Ensure same column order
    synth_encoded = synth_encoded[real_encoded.columns]

    corr_real = real_encoded.corr(numeric_only=True)
    corr_synth = synth_encoded.corr(numeric_only=True)

    # Align matrices
    corr_real, corr_synth = corr_real.align(corr_synth)

    difference = corr_real - corr_synth

    # Replace NaN values
    difference = difference.fillna(0)

    similarity = (
        1 - np.mean(np.abs(difference.values))
    ) * 100

    plt.figure(figsize=(10,8))

    sns.heatmap(
        difference,
        cmap="RdBu_r",
        center=0,
        annot=False,
        square=True
    )

    plt.title(
        f"{model_name} Correlation Difference",
        fontsize=14,
        fontweight="bold"
    )

    plt.tight_layout()
    plt.show()

    print(f"{model_name} Correlation Similarity Score : {similarity:.2f}%")

In [ ]:
plot_correlation_difference(
    df_real,
    df_synthetic_gaussian,
    "GaussianCopula"
)
plot_correlation_difference(
    df_real,
    df_synthetic_ctgan,
    "CTGAN"
)

plot_correlation_difference(
    df_real,
    df_synthetic_tvae,
    "TVAE"
)

## 7.6 PCA Visualization (Real vs Synthetic)

Visualize the overall data distribution using Principal Component Analysis (PCA).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder

# ------------------------------------------------------------
# Encode categorical columns
# ------------------------------------------------------------

def encode_for_pca(real_df, synth_df):

    real = real_df.copy()
    synth = synth_df.copy()

    for col in real.columns:

        if real[col].dtype == "object" or str(real[col].dtype) == "category":

            le = LabelEncoder()

            real[col] = le.fit_transform(real[col].astype(str))

            mapping = {cls: i for i, cls in enumerate(le.classes_)}

            synth[col] = synth[col].astype(str).map(mapping).fillna(-1)

    return real, synth


# ------------------------------------------------------------
# PCA Plot
# ------------------------------------------------------------

def plot_pca(real_df, synth_df, model_name):

    real, synth = encode_for_pca(real_df, synth_df)

    pca = PCA(n_components=2)

    real_pca = pca.fit_transform(real)

    synth_pca = pca.transform(synth)

    plt.figure(figsize=(8,6))

    plt.scatter(
        real_pca[:,0],
        real_pca[:,1],
        alpha=0.4,
        label="Real Data",
        s=20
    )

    plt.scatter(
        synth_pca[:,0],
        synth_pca[:,1],
        alpha=0.4,
        label="Synthetic Data",
        s=20
    )

    plt.title(
        f"PCA Visualization - {model_name}",
        fontsize=14,
        fontweight="bold"
    )

    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")

    plt.legend()

    plt.tight_layout()

    plt.show()

plot_pca(
    df_real,
    df_synthetic_gaussian,
    "GaussianCopula"
)

plot_pca(
    df_real,
    df_synthetic_ctgan,
    "CTGAN"
)

plot_pca(
    df_real,
    df_synthetic_tvae,
    "TVAE"
)

print("\n✅ PCA Completed Successfully!")

## 7.7 ROC Curve Analysis

Compare the classification performance using Receiver Operating Characteristic (ROC) curves.

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(8,6))

for m in models_info:

    fpr, tpr, _ = roc_curve(
        m["y_test"],
        m["probs"]
    )

    plt.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f'{m["name"]} (AUC={m["roc_auc"]:.3f})'
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve Comparison")

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.show()

## 7.8 Precision–Recall Curve Analysis

Evaluate model performance using Precision–Recall curves, particularly for class-wise performance assessment.

In [ ]:
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

for m in models_info:

    precision, recall_vals, _ = precision_recall_curve(
        m["y_test"],
        m["probs"]
    )

    plt.plot(
        recall_vals,
        precision,
        linewidth=2,
        label=f'{m["name"]} (PR-AUC={m["pr_auc"]:.3f})'
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve Comparison")

plt.xlim([0,1])
plt.ylim([0,1])

plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

## 7.9 SHAP Feature Importance Analysis

Interpret feature importance using SHAP values to compare the influence of input features across datasets.

In [ ]:
import shap
import matplotlib.pyplot as plt

for m in models_info:

    print(f"\n===== SHAP Feature Importance - {m['name']} =====")

    explainer = shap.TreeExplainer(m["classifier"])

    shap_values = explainer.shap_values(m["X_test"])

    plt.figure(figsize=(10,6))

    shap.summary_plot(
        shap_values,
        m["X_test"],
        show=False
    )

    plt.title(f"SHAP Summary Plot - {m['name']}")

    plt.tight_layout()

    plt.show()

## 8 t-SNE Visualization

Visualize high-dimensional relationships using t-SNE to compare real and synthetic samples.

In [ ]:

from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import pandas as pd

def plot_tsne(real_df, synth_df, model_name):

    real = real_df.copy()
    synth = synth_df.copy()

    # Encode categorical columns
    for col in real.select_dtypes(include=['object']).columns:
        le = LabelEncoder()

        real[col] = le.fit_transform(real[col].astype(str))

        synth[col] = synth[col].astype(str).map(
            lambda x: le.transform([x])[0] if x in le.classes_ else -1
        )

    # Keep same columns
    synth = synth[real.columns]

    # Sample for faster visualization
    sample_size = min(1000, len(real), len(synth))

    real = real.sample(sample_size, random_state=42)
    synth = synth.sample(sample_size, random_state=42)

    combined = pd.concat([real, synth], ignore_index=True)

    labels = (
        ["Real"] * len(real)
        + ["Synthetic"] * len(synth)
    )

    tsne = TSNE(
        n_components=2,
        random_state=42,
        perplexity=30
    )

    embedding = tsne.fit_transform(combined)

    plt.figure(figsize=(8,6))

    plt.scatter(
        embedding[:len(real),0],
        embedding[:len(real),1],
        alpha=0.5,
        label="Real"
    )

    plt.scatter(
        embedding[len(real):,0],
        embedding[len(real):,1],
        alpha=0.5,
        label="Synthetic"
    )

    plt.title(f"t-SNE Visualization ({model_name})")

    plt.legend()

    plt.tight_layout()

    plt.show()

plot_tsne(df_real, df_synthetic_gaussian, "GaussianCopula")

plot_tsne(df_real, df_synthetic_ctgan, "CTGAN")

plot_tsne(df_real, df_synthetic_tvae, "TVAE")

# 9. Recommendation System

## 9.1 Recommendation Score Comparison

Compare the recommendation scores calculated from multiple evaluation metrics.

In [ ]:

def compute_recommendation(models_info):

    recommendations = []

    for model in models_info:

        quality = model.get("quality_score", 0)
        privacy_nn = model.get("privacy_nn", 0)
        drift = model.get("drift_score", 0)

        accuracy = model.get("accuracy", 0)
        f1 = model.get("f1_score", 0)
        recall = model.get("recall", 0)

        roc_auc = model.get("roc_auc", 0)
        pr_auc = model.get("pr_auc", 0)

        training_time = model.get("training_time", 0)
        memory_usage = model.get("memory_usage", 0)

        # -------------------------
        # Convert "Lower is Better"
        # -------------------------

        privacy_score = 1 / (1 + privacy_nn)
        drift_score = 1 / (1 + drift)

        train_score = 1 / (1 + training_time)
        memory_score = 1 / (1 + memory_usage)

        # -------------------------
        # Final Recommendation Score
        # -------------------------

        recommendation_score = (

            0.22 * quality +

            0.12 * privacy_score +

            0.10 * drift_score +

            0.12 * accuracy +

            0.12 * f1 +

            0.08 * recall +

            0.12 * roc_auc +

            0.07 * pr_auc +

            0.03 * train_score +

            0.02 * memory_score

        )

        recommendations.append({

            "model": model["name"],

            "recommendation_score": recommendation_score,

            "quality": quality,

            "privacy_nn": privacy_nn,

            "drift": drift,

            "accuracy": accuracy,

            "f1": f1,

            "recall": recall,

            "roc_auc": roc_auc,

            "pr_auc": pr_auc,

            "training_time": training_time,

            "memory_usage": memory_usage

        })

    recommendations.sort(
        key=lambda x: x["recommendation_score"],
        reverse=True
    )

    return recommendations

## 9.2 Recommendation Summary Table

Summarize all evaluation metrics and recommendation scores for each synthesizer.

In [ ]:
recommendations = compute_recommendation(models_info)

print("\n===============================")
print(" MODEL RECOMMENDATION RANKING ")
print("===============================\n")

for i, r in enumerate(recommendations, start=1):

    print(f"Rank {i}")

    print(f"Model            : {r['model']}")
    print(f"Recommendation   : {r['recommendation_score']:.3f}")

    print(f"Quality Score    : {r['quality']:.3f}")
    print(f"Privacy NN       : {r['privacy_nn']:.3f}")
    print(f"Drift Score      : {r['drift']:.3f}")

    print(f"Accuracy         : {r['accuracy']:.3f}")
    print(f"F1 Score         : {r['f1']:.3f}")
    print(f"Recall           : {r['recall']:.3f}")

    print(f"ROC-AUC          : {r['roc_auc']:.3f}")
    print(f"PR-AUC           : {r['pr_auc']:.3f}")

    print(f"Training Time    : {r['training_time']:.2f} sec")
    print(f"Memory Usage     : {r['memory_usage']:.2f} MB")

    print("-" * 60)

best_model = recommendations[0]

print("\n====================================")
print(" FINAL RECOMMENDED MODEL")
print("====================================")

print(f"Model : {best_model['model']}")
print(f"Overall Recommendation Score : {best_model['recommendation_score']:.3f}")

## 9.3 Final Model Recommendation Ranking Score

Identify the most suitable synthetic data generation model based on comprehensive evaluation.

In [ ]:


import matplotlib.pyplot as plt

recommendation_models = [r["model"] for r in recommendations]
recommendation_scores = [r["recommendation_score"] for r in recommendations]

plt.figure(figsize=(8,5))

bars = plt.bar(
    recommendation_models,
    recommendation_scores
)

plt.title("Final Model Recommendation Ranking", fontsize=14)
plt.xlabel("Models")
plt.ylabel("Recommendation Score")

# Display score on top of each bar
for bar, score in zip(bars, recommendation_scores):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height()+0.003,
        f"{score:.3f}",
        ha='center',
        fontsize=10
    )

plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 9.4 Final Model Recommendation

Identify the most suitable synthetic data generation model based on comprehensive evaluation.

In [ ]:
best = recommendations[0]

print("="*60)
print("🏆 FINAL RECOMMENDED MODEL")
print("="*60)

print(f"Model Name           : {best['model']}")
print(f"Recommendation Score : {best['recommendation_score']:.3f}")

print("\nReason:")

print("• High synthetic data quality")
print("• Strong machine learning utility")
print("• Good privacy preservation")
print("• Acceptable data drift")
print("• Excellent ROC-AUC / PR-AUC")
print("• Balanced overall performance")

## 9.5 Overall Model Performance Comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

metrics = [
    "Quality",
    "Privacy",
    "Accuracy",
    "F1",
    "Recall",
    "ROC-AUC",
    "PR-AUC"
]

fig = plt.figure(figsize=(8,8))
ax = plt.subplot(111, polar=True)

angles = np.linspace(
    0,
    2*np.pi,
    len(metrics),
    endpoint=False
).tolist()

angles += angles[:1]

for m in models_info:

    privacy_score = 1/(1+m["privacy_nn"])

    values = [
        m["quality_score"],
        privacy_score,
        m["accuracy"],
        m["f1_score"],
        m["recall"],
        m["roc_auc"],
        m["pr_auc"]
    ]

    values += values[:1]

    ax.plot(
        angles,
        values,
        linewidth=2,
        label=m["name"]
    )

    ax.fill(
        angles,
        values,
        alpha=0.10
    )

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics)

plt.title("Overall Model Performance Comparison")

plt.legend(loc="upper right")

plt.show()

## 9.6 Project Conclusion model table

Summarize the overall findings and performance of the implemented synthetic data generation models.

In [ ]:
import pandas as pd

recommendation_df = pd.DataFrame(recommendations)

display(
    recommendation_df[
        [
            "model",
            "recommendation_score",
            "quality",
            "privacy_nn",
            "drift",
            "accuracy",
            "f1",
            "recall",
            "roc_auc",
            "pr_auc"
        ]
    ].sort_values(
        "recommendation_score",
        ascending=False
    )
)

# 10. Conclusion

## 10.1 Project Conclusion

This project successfully developed a privacy-preserving synthetic data generation framework using three SDV synthesizers: GaussianCopula, CTGAN, and TVAE. Synthetic datasets were generated from the Adult Income dataset and evaluated using multiple quality, privacy, statistical similarity, machine learning utility, and data drift metrics.

The comparative analysis showed that each model has different strengths, with the final recommendation based on the combined evaluation of all performance metrics. The project demonstrates that synthetic data can effectively preserve the statistical characteristics and analytical usefulness of the original dataset while reducing privacy risks, making it suitable for data sharing, research, and machine learning applications.

## 11 Future Scope

Future enhancements to this project may include:

- Evaluate additional synthetic data generation models beyond the current SDV synthesizers.
- Perform advanced privacy risk assessments using membership inference and attribute inference attacks.
- Validate the framework on larger and domain-specific datasets such as healthcare, finance, and cybersecurity.
- Develop a web-based application for automated synthetic data generation and evaluation.
- Extend the framework to support time-series and relational datasets.
- Optimize model training for improved scalability and faster synthetic data generation.



## 12 Author Contributions

This project was carried out through collaborative efforts, with clearly defined roles and responsibilities assigned to each team member.

#### Rakshitha G

>1. Conceptualized and implemented the complete synthetic data generation and evaluation pipeline in Jupyter Notebook.
2. Developed modules for structural alignment, quality assessment, diagnostic testing, drift analysis, privacy evaluation, and machine learning utility validation.
3. Designed and implemented the Streamlit-based deployment interface.
4. Integrated all components into a cohesive and functional system.

#### Chinthana P

>1. Assisted in the preparation of the project synopsis.
3. Supported documentation and structural organization of the project framework.

#### Haritha

>1. Prepared the comprehensive project report.
2. Contributed to synopsis development.
3. Supported research paper drafting and publication activities.

---

# Thank You

Thank you for reviewing this project.

For additional details, please refer to the GitHub repository and project documentation.